# Test GPR-Matern: single problem plot

Colab-ready full notebook. It runs one selected problem/seed and shows the seed-1 objective-space plot with `f_sur`, `f_real`, and `HV_xy_shift`.


### **Package**



In [ ]:
try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')

import importlib
import subprocess
import sys
sys.dont_write_bytecode = True
import warnings

print(sys.version)

DEPENDENCIES = {
    'pymoo': {
        'pip': 'pymoo==0.6.1.6',
        'checks': ('pymoo', 'pymoo.gradient.toolbox', 'pymoo.core.problem', 'pymoo.operators.sampling.lhs'),
        'pip_args': ('--force-reinstall',),
    },
    'GPy': {
        'pip': 'GPy',
        'checks': ('GPy',),
    },
    'yaml': {
        'pip': 'pyyaml',
        'checks': ('yaml',),
    },
    'matplotlib': {
        'pip': 'matplotlib',
        'checks': ('matplotlib',),
    },
}

def first_failed_import(modules):
    for module_name in modules:
        try:
            importlib.import_module(module_name)
        except ImportError as err:
            return module_name, err
    return None, None

def install_dependency(package_name, config):
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        *config.get("pip_args", ()),
        config["pip"],
    ]
    print(f"Installing/updating {package_name}: {config['pip']}")
    subprocess.check_call(command)

packages_to_install = []
for package_name, config in DEPENDENCIES.items():
    failed_module, error = first_failed_import(config["checks"])
    if failed_module is None:
        print(f"{package_name} is available.")
    else:
        print(f"{package_name} check failed at {failed_module}: {error}")
        packages_to_install.append(package_name)

for package_name in packages_to_install:
    install_dependency(package_name, DEPENDENCIES[package_name])

if packages_to_install:
    raise RuntimeError(
        "Packages were installed or repaired. Restart the Colab runtime, "
        "then run again from this package cell before continuing."
    )

warnings.filterwarnings("ignore", message=".*load_learner.*pickle.*")


from pathlib import Path
import contextlib
import io

code_path = Path('/content/drive/MyDrive/2026 Indicator_misleading/src')
repo_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent, code_path.parent, code_path]

for repo_root in repo_candidates:
    if (repo_root / 'src').exists() and str(repo_root) not in sys.path:
        sys.path.append(str(repo_root))

import yaml
import numpy as np
from pymoo.operators.sampling.lhs import LHS

from src.data import generate_data
from src.experiment import (
    compute_surrogate_test_mse,
    format_percent,
    compute_R_indicator,
    run_experiment,
    wilcoxon_gap_test,
)
from src.metrics import get_metrics, get_problem_clip_min
from src.models import GPR_Matern, gpr_pred_mean_std, train_gpr_matern_for_calibration
from src.opt_problem import build_problem
from src.other_functions import mean_std
from src.plotting import (
    append_exp1_4_compact_summary_outputs,
    append_result_summary_outputs,
    evaluate_majority_shift_y_k1_diagnostic_items,
    print_exp1_4_gap_improvement_tables,
    build_bluebear_seed_records,
    summarize_result_records,
    write_result_summary_temp,
)
from src.survival import Survival_dual_ranking, Survival_standard, find_upper_alpha

warnings.filterwarnings("ignore", message=".*load_learner.*pickle.*")
np.set_printoptions(precision=4, suppress=True)



### **Main**



###### 1. Initial settings



In [ ]:
CONFIG_FILE_NAME = "config.yaml"


def _resolve_experiment_config_path(filename=CONFIG_FILE_NAME):
    roots = [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
    ]
    try:
        code_root = Path(code_path)
        roots.extend([code_root.parent, code_root])
    except NameError:
        pass

    seen = set()
    for root in roots:
        for candidate in (root / "experiments" / filename, root / filename):
            candidate = candidate.resolve()
            if candidate in seen:
                continue
            seen.add(candidate)
            if candidate.exists():
                return candidate

    searched = "\n".join(str(path) for path in sorted(seen))
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")


config_path = _resolve_experiment_config_path()
with open(config_path, "r", encoding="utf-8") as config_file:
    experiment_config = yaml.safe_load(config_file)

problem_names = experiment_config["problem_names"]
dtlz_n_var = experiment_config["dtlz_n_var"]
omnitest_n_var = experiment_config["omnitest_n_var"]
dtlz_n_obj = experiment_config["dtlz_n_obj"]
n_gen = experiment_config["n_gen"]
pop_size = experiment_config["pop_size"]
seed_start = experiment_config["seed_start"]
seed_end = experiment_config["seed_end"]
train_seed = experiment_config["train_seed"]
test_seed = experiment_config["test_seed"]
sample_size = experiment_config["sample_size"]
val_size = experiment_config["val_size"]
test_size = experiment_config["test_size"]
calibration_k = experiment_config["calibration_k"]
knn_threshold_method = experiment_config["knn_threshold_method"]
show_seed_output = experiment_config["show_seed_output"]
optimizer_run_specs = experiment_config["optimizer_run_specs"]
optimizer_names = [spec["result_name"] for spec in optimizer_run_specs]
dual_ranking_target_coverage = experiment_config.get("dual_ranking_target_coverage", 0.90)
dual_ranking_alpha_max = experiment_config.get("dual_ranking_alpha_max", 500.0)
dual_ranking_alpha_step = experiment_config.get("dual_ranking_alpha_step", 0.01)
surrogate_config = experiment_config["surrogates"]["gpr_matern"]
method_name = "GPR_Matern"

print(f"Loaded experiment config: {config_path}")
print(f"Problems: {len(problem_names)} | seeds: range({seed_start}, {seed_end}) | n_gen: {n_gen} | pop_size: {pop_size}")
print(f"Optimizers: {optimizer_names}")



###### 2. Surrogate model and summary functions



In [ ]:
use_surrogate = surrogate_config["use_surrogate"]


def reset_experiment_random_state(seed, label=None):
    seed = int(seed)
    np.random.seed(seed)
def train_model_for_calibration(problem, sample_size, train_seed=42, test_seed=1):
    return train_gpr_matern_for_calibration(
        problem=problem,
        sample_size=sample_size,
        train_seed=train_seed,
        test_seed=test_seed,
        val_size=val_size,
        test_size=test_size,
    )


In [ ]:
def build_benchmark_problem(problem_name, dtlz_n_var=10, dtlz_n_obj=2, omnitest_n_var=2):
    pname = problem_name.lower()

    if pname.startswith("dtlz"):
        problem = build_problem(problem_name=problem_name, n_var=dtlz_n_var, n_obj=dtlz_n_obj)
    elif pname == "omnitest":
        problem = build_problem(problem_name=problem_name, n_var=omnitest_n_var)
    else:
        problem = build_problem(problem_name=problem_name)

    return problem



def run_problem(problem_name):
    reset_experiment_random_state(train_seed, f"{problem_name} surrogate/data")
    problem = build_benchmark_problem(
        problem_name,
        dtlz_n_var=dtlz_n_var,
        dtlz_n_obj=dtlz_n_obj,
        omnitest_n_var=omnitest_n_var,
    )
    hv, igd_plus, obj_min, obj_max, _ = get_metrics(
        problem_name=problem_name,
        problem=problem,
        n_var=problem.n_var,
        n_obj=problem.n_obj,
    )

    current_sample_size = sample_size if sample_size is not None else max(11 * problem.n_var - 1, 100)
    problem_y_min = get_problem_clip_min(problem_name, n_var=problem.n_var)
    if problem_y_min is None:
        problem_y_min = obj_min

    model_f1, model_f2, X_train, y_train, f_train_mean, X_val, y_val, X_test, y_test = train_model_for_calibration(
        problem=problem,
        sample_size=current_sample_size,
        train_seed=train_seed,
        test_seed=test_seed,
    )


    offline_test_mse = compute_surrogate_test_mse(
        problem=problem,
        problem_name=problem_name,
        model_f1=model_f1,
        model_f2=model_f2,
        use_surrogate=use_surrogate,
        x_test=X_test,
        y_test=y_test,
    )

    diagnostic_context = {
        'X_train': X_train,
        'y_train': y_train,
        'f_train_mean': f_train_mean,
        'hv': hv,
        'obj_min': obj_min,
        'obj_max': obj_max,
        'problem_y_min': problem_y_min,
        'offline_test_mse': offline_test_mse,
    }
    dual_ranking_survival = None
    for optimizer_spec in optimizer_run_specs:
        result_name = optimizer_spec["result_name"]
        optimizer_name = optimizer_spec["optimizer_name"]
        if optimizer_spec["use_dual_ranking"]:
            if dual_ranking_survival is None:
                dual_ranking_alpha_f1, dual_ranking_coverage_f1 = find_upper_alpha(
                    model_f1,
                    X_val,
                    y_val[:, 0],
                    target_coverage=dual_ranking_target_coverage,
                    alpha_max=dual_ranking_alpha_max,
                    alpha_step=dual_ranking_alpha_step,
                )
                dual_ranking_alpha_f2, dual_ranking_coverage_f2 = find_upper_alpha(
                    model_f2,
                    X_val,
                    y_val[:, 1],
                    target_coverage=dual_ranking_target_coverage,
                    alpha_max=dual_ranking_alpha_max,
                    alpha_step=dual_ranking_alpha_step,
                )
                print(
                    f"Dual-ranking upper bounds: "
                    f"alpha_f1={dual_ranking_alpha_f1:.2f} (coverage={dual_ranking_coverage_f1:.1%}), "
                    f"alpha_f2={dual_ranking_alpha_f2:.2f} (coverage={dual_ranking_coverage_f2:.1%})"
                )
                dual_ranking_survival = Survival_dual_ranking(
                    alpha_f1=dual_ranking_alpha_f1,
                    alpha_f2=dual_ranking_alpha_f2,
                )
            survival_function = dual_ranking_survival
        else:
            survival_function = Survival_standard()

        run_kwargs = dict(
            problem=problem,
            problem_name=problem_name,
            n_gen=n_gen,
            pop_size=pop_size,
            model_f1=model_f1,
            model_f2=model_f2,
            obj_min=obj_min,
            obj_max=obj_max,
            hv=hv,
            igd_plus=igd_plus,
            use_surrogate=use_surrogate,
            survival_function=survival_function,
            use_callback=False,
            seeds=range(seed_start, seed_end),
            optimizer_name=optimizer_name,
            distance_x=X_train,
            distance_y=y_train,
            problem_y_min=problem_y_min,
            knn_distance_space="x",
            f_train_mean=f_train_mean,
            calibration_k=calibration_k,
            knn_threshold_method=knn_threshold_method,
            compact_seed_output=True,
            compact_seed_context=diagnostic_context,
            compact_seed_mse_test=offline_test_mse,
            solution_output_dir=Path(config_path).resolve().parent / "output",
            solution_output_method_name=f"{method_name}+{result_name}",
            compact_seed_method_name=f"{method_name}_{result_name}",
        )

        results = run_experiment(**run_kwargs)

        try:
            y_space_diagnostic_items = evaluate_majority_shift_y_k1_diagnostic_items(
                results,
                diagnostic_context,
            )
            print(f"\n============================== {problem_name} | {result_name} ==============================")
            seed_records = build_bluebear_seed_records(
                results,
                diagnostic_context,
                mse_test=offline_test_mse,
                y_space_items=y_space_diagnostic_items,
            )
            result_summary_rows.append(
                summarize_result_records(
                    exp_name=method_name,
                    method_name=method_name,
                    problem_name=problem_name,
                    optimizer_name=result_name,
                    records=seed_records,
                    output_dir=Path(config_path).resolve().parent / "output",
                )
            )
            write_result_summary_temp(
                result_summary_rows,
                output_dir=Path(config_path).resolve().parent / "output",
                method_name=method_name,
            )
            del seed_records, y_space_diagnostic_items, results
        except Exception as err:
            print(f"\n============================== {problem_name} | {result_name} ==============================")
            print(f"Problem finished, but summary printing failed: {type(err).__name__}: {err}")
            print(f"Available result keys: {sorted(results.keys())}")
            raise

    del model_f1, model_f2, X_train, y_train, f_train_mean, X_val, y_val, X_test, y_test
    return None


### Single test run


In [ ]:
# ===== Test controls: run one selected problem, all optimizers, one seed, and draw xy-shift plots =====
SELECTED_PROBLEM = "dtlz1"
SELECTED_SEED = 1

import os
from pathlib import Path
os.environ.pop("DISABLE_HV_PLOTS", None)
os.environ["HV_TEST_PLAIN_PLOT"] = "1"
os.environ["HV_TEST_SAVE_SVG"] = "1"
os.environ["HV_TEST_SVG_PREFIX"] = "negative1"
os.environ["HV_TEST_SEPARATE_LEGEND"] = "1"
os.environ["HV_TEST_SVG_DIR"] = str(Path("experiments/plot_sur_real/svg").resolve())

problem_names = [SELECTED_PROBLEM]
seed_start = int(SELECTED_SEED)
seed_end = int(SELECTED_SEED) + 1
optimizer_run_specs = list(experiment_config["optimizer_run_specs"])
optimizer_names = [spec["result_name"] for spec in optimizer_run_specs]

print(
    f"Test run | problem={SELECTED_PROBLEM} | "
    f"methods={optimizer_names} | seed={SELECTED_SEED}"
)

all_results = {}
problem_contexts = {}
result_summary_rows = []

for problem_index, problem_name in enumerate(problem_names, start=1):
    result = run_problem(problem_name)
    if result is not None:
        all_results[problem_name] = result


In [ ]:
# Aggregate gap-improvement table printing is skipped for this single plotting test.
gap_improvement_tables = None


In [ ]:
# Aggregate result-file writing is skipped for this single plotting test.
# Per-seed values and the xy-shift plot are printed during run_problem.
result_summary_table = None
